# Module 9 • Machine Translation

# Lesson 52 • Neural Machine Translation with Encoder–Decoder Networks and Attention

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU only

## Scope

This lesson moves from statistical MT to neural sequence-to-sequence translation. It covers encoder–decoder modeling, teacher forcing, attention, training, greedy decoding, beam-search intuition, evaluation, and Arabic-specific considerations. The executable core trains a tiny attention-based NMT system fully offline on CPU.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain sequence-to-sequence NMT;
- distinguish encoder and decoder roles;
- build source and target vocabularies;
- explain teacher forcing and exposure bias;
- implement additive attention;
- train a tiny encoder–decoder model;
- decode translations greedily;
- interpret attention weights cautiously;
- explain beam search and length bias;
- evaluate translation outputs;
- discuss subword modeling, Arabic morphology, and tashkeel preservation.

## Table of Contents

1. From SMT to NMT  
2. Sequence-to-Sequence Learning  
3. Encoder–Decoder Architecture  
4. Teacher Forcing  
5. Cross-Entropy Objective  
6. Fixed-Vector Bottleneck  
7. Attention  
8. Parallel Corpus  
9. Vocabulary and Numericalization  
10. Padding and Batching  
11. Encoder  
12. Additive Attention  
13. Decoder  
14. Seq2Seq Model  
15. Training  
16. Training Curve  
17. Greedy Decoding  
18. Translation Examples  
19. Attention Inspection  
20. Beam Search  
21. Exposure and Length Bias  
22. Subword Tokenization  
23. Arabic NMT  
24. Evaluation  
25. Error Analysis  
26. SMT vs NMT  
27. Limitations of RNN NMT  
28. Reproducibility  
29. Knowledge Check  
30. Exercises  
31. Summary and Next Lesson

# 1. From SMT to NMT

Statistical MT combines separately estimated translation, language, and reordering models. Neural MT instead trains a single differentiable model end to end.

In [ ]:
import math
import platform
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")

pd.DataFrame([
    ("SMT", "phrase tables + language model + decoder"),
    ("NMT", "single end-to-end neural model"),
], columns=["Approach", "Core architecture"])

# 2. Sequence-to-Sequence Learning

A sequence-to-sequence model maps a variable-length source sequence to a variable-length target sequence. In translation, the source and target lengths usually differ.

# 3. Encoder–Decoder Architecture

The **encoder** maps the source sentence to hidden representations. The **decoder** generates the target sentence autoregressively, one token at a time.

# 4. Teacher Forcing

During training, teacher forcing supplies the correct previous target token rather than the decoder's own prediction. This stabilizes optimization but creates a train–inference mismatch.

# 5. Cross-Entropy Objective

At each target position, the decoder predicts a distribution over the target vocabulary. Cross-entropy penalizes low probability assigned to the correct next token.

# 6. Fixed-Vector Bottleneck

Early encoder–decoder models compressed the full source sentence into one vector. Longer sentences degraded because too much information had to pass through a single representation.

# 7. Attention

Attention lets each decoder step access all encoder hidden states. Alignment scores are normalized into attention weights, which produce a context vector as a weighted sum of source representations.

# 8. Parallel Corpus

In [ ]:
parallel_pairs = [
    ("je suis ici", "i am here"),
    ("je suis petit", "i am small"),
    ("je suis grand", "i am big"),
    ("tu es ici", "you are here"),
    ("tu es petit", "you are small"),
    ("tu es grand", "you are big"),
    ("il est ici", "he is here"),
    ("il est petit", "he is small"),
    ("il est grand", "he is big"),
    ("elle est ici", "she is here"),
    ("elle est petite", "she is small"),
    ("elle est grande", "she is big"),
    ("nous sommes ici", "we are here"),
    ("vous etes ici", "you are here"),
]
pd.DataFrame(parallel_pairs, columns=["French", "English"])

# 9. Vocabulary and Numericalization

In [ ]:
SPECIAL = ["<pad>", "<sos>", "<eos>", "<unk>"]
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

def build_vocab(sentences):
    words = sorted({w for s in sentences for w in s.split()})
    itos = SPECIAL + words
    stoi = {w:i for i,w in enumerate(itos)}
    return stoi, itos

src_sentences = [s for s,_ in parallel_pairs]
tgt_sentences = [t for _,t in parallel_pairs]
src_stoi, src_itos = build_vocab(src_sentences)
tgt_stoi, tgt_itos = build_vocab(tgt_sentences)


def encode_sentence(sentence, stoi):
    ids = [SOS_IDX] + [stoi.get(w, UNK_IDX) for w in sentence.split()] + [EOS_IDX]
    return torch.tensor(ids, dtype=torch.long)

print("Source vocab:", len(src_itos))
print("Target vocab:", len(tgt_itos))
print(encode_sentence("je suis ici", src_stoi))

# 10. Padding and Batching

Padding creates rectangular mini-batches while a mask prevents the attention mechanism from attending to padding positions.

In [ ]:
src_tensors = [encode_sentence(s, src_stoi) for s in src_sentences]
tgt_tensors = [encode_sentence(t, tgt_stoi) for t in tgt_sentences]
SOURCE_BATCH = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX).to(DEVICE)
TARGET_BATCH = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX).to(DEVICE)
print(SOURCE_BATCH.shape, TARGET_BATCH.shape)

# 11. Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, source):
        embedded = self.embedding(source)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

# 12. Additive Attention

Bahdanau-style additive attention learns a nonlinear compatibility function between the decoder state and every encoder state.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.energy = nn.Linear(hidden_dim * 2, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask):
        src_len = encoder_outputs.size(1)
        repeated = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.energy(torch.cat([repeated, encoder_outputs], dim=2)))
        scores = self.score(energy).squeeze(2)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights

# 13. Decoder

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = attention
        self.rnn = nn.GRU(emb_dim + hidden_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim + hidden_dim + emb_dim, vocab_size)

    def forward(self, input_token, hidden, encoder_outputs, mask):
        embedded = self.embedding(input_token)
        context, weights = self.attention(hidden[-1], encoder_outputs, mask)
        rnn_input = torch.cat([embedded, context], dim=1).unsqueeze(1)
        rnn_output, hidden = self.rnn(rnn_input, hidden)
        rnn_output = rnn_output.squeeze(1)
        logits = self.out(torch.cat([rnn_output, context, embedded], dim=1))
        return logits, hidden, weights

# 14. Seq2Seq Model

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_forcing_ratio=1.0):
        batch_size, tgt_len = target.shape
        vocab_size = self.decoder.out.out_features
        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=source.device)
        encoder_outputs, hidden = self.encoder(source)
        mask = source != PAD_IDX
        input_token = target[:, 0]

        for step in range(1, tgt_len):
            logits, hidden, _ = self.decoder(input_token, hidden, encoder_outputs, mask)
            outputs[:, step, :] = logits
            predicted = logits.argmax(dim=1)
            use_teacher = random.random() < teacher_forcing_ratio
            input_token = target[:, step] if use_teacher else predicted

        return outputs

# 15. Training

In [ ]:
EMB_DIM = 32
HIDDEN_DIM = 64
EPOCHS = 100

encoder = Encoder(len(src_itos), EMB_DIM, HIDDEN_DIM)
attention = AdditiveAttention(HIDDEN_DIM)
decoder = Decoder(len(tgt_itos), EMB_DIM, HIDDEN_DIM, attention)
model = Seq2Seq(encoder, decoder).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
losses = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    output = model(SOURCE_BATCH, TARGET_BATCH, teacher_forcing_ratio=0.8)
    logits = output[:, 1:, :].reshape(-1, len(tgt_itos))
    gold = TARGET_BATCH[:, 1:].reshape(-1)
    loss = criterion(logits, gold)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(float(loss.item()))

print("Initial loss:", round(losses[0], 4))
print("Final loss:", round(losses[-1], 4))

# 16. Training Curve

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(range(1, EPOCHS + 1), losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Tiny Attention-Based NMT Training")
plt.tight_layout()
plt.show()

# 17. Greedy Decoding

Greedy decoding selects the highest-probability next token at every step.

In [ ]:
def greedy_translate(sentence, max_length=10):
    model.eval()
    source = encode_sentence(sentence, src_stoi).unsqueeze(0).to(DEVICE)
    mask = source != PAD_IDX

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(source)
        input_token = torch.tensor([SOS_IDX], device=DEVICE)
        generated = []
        attn_history = []

        for _ in range(max_length):
            logits, hidden, weights = model.decoder(input_token, hidden, encoder_outputs, mask)
            next_token = int(logits.argmax(dim=1).item())
            attn_history.append(weights[0].cpu().numpy())
            if next_token == EOS_IDX:
                break
            if next_token not in {PAD_IDX, SOS_IDX}:
                generated.append(tgt_itos[next_token])
            input_token = torch.tensor([next_token], device=DEVICE)

    return " ".join(generated), np.array(attn_history)

print(greedy_translate("je suis petit")[0])

# 18. Translation Examples

In [ ]:
examples = ["je suis ici", "tu es grand", "elle est petite", "il est petit", "nous sommes ici"]
reference_map = dict(parallel_pairs)
rows = []
for source in examples:
    prediction, _ = greedy_translate(source)
    rows.append({"source": source, "reference": reference_map[source], "prediction": prediction})
pd.DataFrame(rows)

# 19. Attention Inspection

Attention matrices can resemble soft alignments, but attention weights should not automatically be treated as causal explanations.

In [ ]:
source_sentence = "je suis petit"
prediction, weights = greedy_translate(source_sentence)
source_tokens = ["<sos>"] + source_sentence.split() + ["<eos>"]
target_tokens = prediction.split()
attention_frame = pd.DataFrame(
    weights[:len(target_tokens), :len(source_tokens)],
    index=target_tokens,
    columns=source_tokens,
)
attention_frame

# 20. Beam Search

Beam search keeps multiple partial hypotheses instead of only one. It can recover a better total sequence than greedy decoding, but it requires more computation and often needs length normalization.

In [ ]:
pd.DataFrame([
    (1, "i", -0.2),
    (1, "you", -0.5),
    (2, "i am", -0.3),
    (2, "you are", -0.7),
    (2, "i are", -1.6),
], columns=["Step", "Hypothesis", "Log score"])

# 21. Exposure and Length Bias

**Exposure bias** comes from training with gold previous tokens while inference uses model-generated history. **Length bias** arises because sequence scores accumulate across decoding steps, which can favor particular output lengths.

# 22. Subword Tokenization

Modern NMT commonly uses BPE, SentencePiece, or unigram subword tokenization. Subwords reduce unknown-word problems and help with rare morphology.

# 23. Arabic NMT

Arabic NMT must handle rich morphology, clitics, orthographic variation, optional tashkeel, flexible word order, and dialect variation.

In [ ]:
pd.DataFrame([
    ("وَسَيَكْتُبُونَهَا", "complex inflected verb"),
    ("بِالْمَدْرَسَةِ", "preposition + noun"),
    ("كِتَابُهُمَا", "noun + dual possessive suffix"),
], columns=["Arabic form", "NMT challenge"])

For **fully vocalized Arabic** experiments, tashkeel must be preserved consistently in training data, tokenizer input, reference translations, generated output, and evaluation. Removing tashkeel changes the experimental task.

# 24. Evaluation

NMT evaluation should combine automatic metrics with linguistic error analysis. Common metrics include BLEU, chrF, COMET, and semantic similarity measures.

In [ ]:
def token_accuracy(reference, prediction):
    ref = reference.split()
    pred = prediction.split()
    if not ref:
        return 0.0
    correct = sum(a == b for a,b in zip(ref,pred))
    return correct / len(ref)

rows = []
for source, reference in parallel_pairs:
    prediction, _ = greedy_translate(source)
    rows.append({
        "source": source,
        "reference": reference,
        "prediction": prediction,
        "exact": prediction == reference,
        "token_accuracy": token_accuracy(reference, prediction),
    })

evaluation_frame = pd.DataFrame(rows)
pd.Series({
    "Exact sequence accuracy": float(evaluation_frame["exact"].mean()),
    "Mean token accuracy": float(evaluation_frame["token_accuracy"].mean()),
})

# 25. Error Analysis

In [ ]:
pd.DataFrame([
    ("Lexical", "wrong translated word"),
    ("Agreement", "incorrect grammatical agreement"),
    ("Word order", "incorrect target order"),
    ("Omission", "source content missing"),
    ("Addition", "unsupported content added"),
    ("Termination", "decoder stops too early or too late"),
], columns=["Error type", "Description"])

# 26. SMT vs NMT

In [ ]:
pd.DataFrame([
    ("Representation", "discrete phrase tables", "continuous embeddings"),
    ("Context", "local phrases + LM", "distributed sentence context"),
    ("Training", "multiple modules", "end-to-end"),
    ("Rare words", "phrase-table coverage", "subword modeling"),
    ("Interpretability", "explicit phrase/alignment tables", "less directly interpretable"),
], columns=["Dimension", "SMT", "NMT"])

# 27. Limitations of RNN NMT

RNN-based NMT is sequential and can struggle with very long dependencies. Attention improved information flow, but Transformer MT removed recurrence and enabled more parallel computation.

# 28. Reproducibility

In [ ]:
pd.Series({
    "module": "Module 9 • Machine Translation",
    "lesson": "Lesson 52",
    "sentence_pairs": len(parallel_pairs),
    "source_vocab": len(src_itos),
    "target_vocab": len(tgt_itos),
    "embedding_dim": EMB_DIM,
    "hidden_dim": HIDDEN_DIM,
    "epochs": EPOCHS,
    "device": str(DEVICE),
    "seed": SEED,
    "python": platform.python_version(),
    "torch": torch.__version__,
}, name="Lesson 52 experiment")

# 29. Knowledge Check

1. What does the encoder produce?  
2. What does the decoder predict?  
3. What is teacher forcing?  
4. Why is cross-entropy used?  
5. What problem does attention solve?  
6. What is a context vector?  
7. How does greedy decoding work?  
8. How does beam search differ?  
9. What is exposure bias?  
10. Why are subwords useful?  
11. Why is Arabic challenging for NMT?  
12. Why preserve tashkeel in fully vocalized tasks?  
13. What errors should MT evaluation inspect?  
14. How does NMT differ structurally from SMT?  
15. Why did Transformer MT replace recurrent NMT in many systems?

# 30. Exercises

## Exercise 1
Add more French–English sentence pairs.

## Exercise 2
Train without attention and compare results.

## Exercise 3
Compare multiple teacher-forcing ratios.

## Exercise 4
Increase hidden size and measure training effects.

## Exercise 5
Add dropout.

## Exercise 6
Implement beam search.

## Exercise 7
Plot a separate attention heatmap figure.

## Exercise 8
Replace words with subword tokens.

## Exercise 9
Build a tiny fully vocalized Arabic–English corpus.

## Exercise 10
Compare exact match with BLEU and chrF.

## Challenge Exercises

1. Add a bidirectional encoder.  
2. Use packed sequences for variable-length mini-batches.  
3. Implement beam search with length normalization.  
4. Train Arabic→English and English→Arabic toy models.  
5. Compare this recurrent NMT system with Transformer MT.

# 31. Summary and Next Lesson

In this lesson, a complete tiny attention-based NMT system was implemented and trained on CPU. The lesson covered encoder–decoder modeling, teacher forcing, additive attention, greedy decoding, beam-search concepts, exposure bias, subword tokenization, evaluation, Arabic morphology, and tashkeel preservation.

## Next Lesson

**Lesson 53: Transformer-Based Machine Translation and Pretrained Multilingual Models** introduces Transformer MT, encoder–decoder self-attention, MarianMT, M2M-100, NLLB-style multilingual translation, and modern pretrained workflows.

# References

- Sutskever, I. et al. *Sequence to Sequence Learning with Neural Networks*.
- Cho, K. et al. work on encoder–decoder neural machine translation.
- Bahdanau, D. et al. *Neural Machine Translation by Jointly Learning to Align and Translate*.
- Luong, M.-T. et al. *Effective Approaches to Attention-based Neural Machine Translation*.
- Koehn, P. *Neural Machine Translation*.